# Data Processing

This notebook is responsible for all data processing and cleaning tasks. Currently, only data sourced from **PokéApi** is processed here.

Data from **Smogon API** is used minimally during this phase. It will be utilized extensively during the **Machine Learning** phase; consequently, the raw Smogon data will be processed later during the feature engineering stage.



In [ ]:
from google.colab import drive
import json
import os

# Mount drive
drive.mount('/content/drive')

# Define paths
project_data_path_raw = '/content/drive/MyDrive/DSA210-Project/data/raw'
project_data_path = '/content/drive/MyDrive/DSA210-Project/data/'

Mounted at /content/drive


In [ ]:
# Load jsons to be processed
with open(os.path.join(project_data_path_raw, 'abilities_raw.json'), 'r') as f:
    abilities_raw = json.load(f)

with open(os.path.join(project_data_path_raw, 'move_damage_classes_raw.json'), 'r') as f:
    move_damage_classes_raw = json.load(f)

with open(os.path.join(project_data_path_raw, 'pokemon_raw.json'), 'r') as f:
    pokemon_raw = json.load(f)

with open(os.path.join(project_data_path_raw, 'types_raw.json'), 'r') as f:
    types_raw = json.load(f)

with open(os.path.join(project_data_path_raw, 'species_raw.json'), 'r') as f:
    species_raw = json.load(f)

with open(os.path.join(project_data_path_raw, 'generations_raw.json'), 'r') as f:
    generations_raw = json.load(f)

## Abilities

### Unnecessary Fields
The following fields will be excluded from the processed JSON:
* `effect_changes`, `effect_entries`, `flavor_text_entries`
* `id`
* `name`, `names`
* `is_main_series`

---

### Reasoning

* **Descriptions:** `effects` and `flavor_text` are descriptions of what an ability does, which falls outside the quantitative scope of this project.
* **Identifiers:** `id` is an internal API identifier. Since ability names are unique, this field is redundant and will be dropped.
* **Name/Names:** `name` is  utilized as the key, making the field itself redundant. `names` contains translations in other languages, which are irrelevant to this analysis.
* **Main Series:** Non-main series content isn't within the scope of this project. `is_main_series` will be utilized during the cleaning logic to filter out non-main-series content. However, once the filtering is complete, the field itself will be removed from the processed data.

In [ ]:
print(abilities_raw["stench"].keys())

dict_keys(['effect_changes', 'effect_entries', 'flavor_text_entries', 'generation', 'id', 'is_main_series', 'name', 'names', 'pokemon'])


In [ ]:
abilities = {}

keys_to_remove_abilities = [
    'effect_changes',
    'effect_entries',
    'flavor_text_entries',
    'id',
    'name',
    'names',
    'is_main_series'
]

for ability_name, details in abilities_raw.items():
    # Check if it's a main series ability
    if details.get('is_main_series') is True:

        cleaned_detail = details.copy()

        # Remove the unwanted keys
        for key in keys_to_remove_abilities:
            cleaned_detail.pop(key, None) # None prevents errors if key is missing

        abilities[ability_name] = cleaned_detail

with open(os.path.join(project_data_path, 'abilities.json'), 'w', encoding='utf-8') as f:
    json.dump(abilities, f, indent=4)

print(f"Cleaning complete!")
print(f"Original items: {len(abilities_raw)}")
print(f"Main Series items: {len(abilities)}")

Cleaning complete!
Original items: 367
Main Series items: 307


## Move Damage Classes

### Unnecessary Fields
The following fields will be excluded from the processed JSON:
* `descriptions`
* `id`
* `name`, `names`

---

### Reasoning

* **Descriptions:** `descriptions` is a text explanation of a move class for example: status which refers to non-damaging moves. This field is unncessary for this project.
* **Identifiers:** `id` is an internal API identifier. Since move class names are unique, this field is redundant and will be dropped.
* **Name/Names:** `name` is  utilized as the key, making the field itself redundant. `names` contains translations in other languages, which are irrelevant to this analysis.


In [ ]:
print(move_damage_classes_raw["status"].keys())

dict_keys(['descriptions', 'id', 'moves', 'name', 'names'])


In [ ]:
move_damage_classes = {}

keys_to_remove_move_damage_classes = [
    'descriptions',
    'id',
    'name',
    'names',
]

for move_class_name, details in move_damage_classes_raw.items():

      cleaned_detail = details.copy()

      # Remove the unwanted keys
      for key in keys_to_remove_move_damage_classes:
          cleaned_detail.pop(key, None) # None prevents errors if key is missing

      move_damage_classes[move_class_name] = cleaned_detail

# the removed last elements here are moves that are not from the main series

status_moves = move_damage_classes["status"]["moves"]
print(status_moves[-1]["name"])
for i in range(6):
  status_moves.pop()
print(status_moves[-1]["name"])

physical_moves = move_damage_classes["physical"]["moves"]
print(physical_moves[-1]["name"])
for i in range(5):
  physical_moves.pop()
print(physical_moves[-1]["name"])

special_moves = move_damage_classes["special"]["moves"]
print(special_moves[-1]["name"])
for i in range(7):
  special_moves.pop()
print(special_moves[-1]["name"])

with open(os.path.join(project_data_path, 'move_damage_classes.json'), 'w', encoding='utf-8') as f:
    json.dump(move_damage_classes, f, indent=4)

print(f"\nCleaning complete!")
print(f"Original fields: {move_damage_classes_raw["status"].keys()}")
print(f"Processed fields: {move_damage_classes["status"].keys()}")


shadow-sky
dragon-cheer
shadow-end
upper-hand
shadow-half
malignant-chain

Cleaning complete!
Original fields: dict_keys(['descriptions', 'id', 'moves', 'name', 'names'])
Processed fields: dict_keys(['moves'])


## Pokémon

### Unnecessary Fields
The following fields will be excluded from the processed JSON:
* `base_experience`
* `id`
* `name`
* `cries`
* `forms`
* `game_indices`
* `height`
* `location_area_encounters`
* `moves`
* `species`
* `sprites`
* `weight`
---

### Reasoning

* **Irrelevant Details:** `base_experience` is used in calculating the amount of experience earned when defeating a pokémon of this species. `height` and `weight` are self-explanatory. These details are irrelvant for the statistical analysis that is performed in this project.
* **Identifiers & Redundancy:** `id` is an internal API index. `name` is used as the unique key for the resource, making both fields redundant in the processed JSON.
* **Aesthetic & Media Assets:** `sprites` (images) and `cries` (audio files) are media assets that cannot be utilized for numerical or categorical analysis in this context.
* **Game and Location Data:** `game_indices` and `location_area_encounters` refer to game and version specific information and spawn locations, which are irrelevant for statistical or competitive analysis.
* **Forms:** `forms` does not provide any meaningful information for this analysis.
* **Species:** Some Pokémon are of the same species but have different variantions such as males and females of the species having different designs, abilities, types or stats. For example, Indeedee and Meowstic. In this project I would like to regard each as a seperate Pokémon because they have differing qualities. Still the species endpoint does have some useful information such as generation, is_legendary, is_baby and is_mythical fields. This field contains a link to the species endpoint for each Pokémon however to present all raw data in a clean format within this project's repository I have chosen to get this information by downloading the species JSON from the species endpoint instead of through this field as such it will be dropped here.
* **Moves:** The `moves` field contains a massive array of every move a Pokémon can learn across all generations. This is not only unnecessary for this project but it also inflates JSON sizes. The only use of moves will be in the Machine Learning phase and that will be focused only on commonly used moves for each Pokémon not their entire move-pool. Information on commonly used moves is available from Smogon API as such this field will be dropped.


In [ ]:
# Unnecessary fields: base_experience cries forms game_indices height held_items id location_area_encounters moves name species sprites weight

print(pokemon_raw["bulbasaur"].keys())

dict_keys(['abilities', 'base_experience', 'cries', 'forms', 'game_indices', 'height', 'held_items', 'id', 'is_default', 'location_area_encounters', 'moves', 'name', 'order', 'past_abilities', 'past_stats', 'past_types', 'species', 'sprites', 'stats', 'types', 'weight'])


In [ ]:
# The species data will be combined with pokemon data, the combined json will inherit the following fields from species:
# evolves_from, generation, is_baby, is_legendary, is_mythical, gender_rate

print(species_raw["bulbasaur"].keys())

dict_keys(['base_happiness', 'capture_rate', 'color', 'egg_groups', 'evolution_chain', 'evolves_from_species', 'flavor_text_entries', 'form_descriptions', 'forms_switchable', 'gender_rate', 'genera', 'generation', 'growth_rate', 'habitat', 'has_gender_differences', 'hatch_counter', 'id', 'is_baby', 'is_legendary', 'is_mythical', 'name', 'names', 'order', 'pal_park_encounters', 'pokedex_numbers', 'shape', 'varieties'])


## Irrelevant Pokémon Cleanup
The following code blocks are the identification and then removal of Pokémon that shouldn't be in the dataset for various reasons (but mostly data duplication) I will provide the detailed rationale for each in the future, for now trust they are excluded for good reason

In [ ]:
for pkmn, details in pokemon_raw.items():
    if('pikachu' in pkmn):
      print(pkmn)

pikachu
pikachu-rock-star
pikachu-belle
pikachu-pop-star
pikachu-phd
pikachu-libre
pikachu-cosplay
pikachu-original-cap
pikachu-hoenn-cap
pikachu-sinnoh-cap
pikachu-unova-cap
pikachu-kalos-cap
pikachu-alola-cap
pikachu-partner-cap
pikachu-starter
pikachu-world-cap
pikachu-gmax


In [ ]:
for pkmn, details in pokemon_raw.items():
    if('greninja' in pkmn):
      print(pkmn)

greninja
greninja-battle-bond
greninja-ash
greninja-mega


In [ ]:
for pkmn, details in pokemon_raw.items():
    if('eevee' in pkmn):
      print(pkmn)

eevee
eevee-starter
eevee-gmax


In [ ]:
for pkmn, details in pokemon_raw.items():
    if('floette' in pkmn):
      print(pkmn)

floette
floette-eternal
floette-mega


In [ ]:
for pkmn, details in pokemon_raw.items():
    if('zygarde' in pkmn):
      print(pkmn)
      print(details.get('abilities'))

zygarde-50
[{'ability': {'name': 'aura-break', 'url': 'https://pokeapi.co/api/v2/ability/188/'}, 'is_hidden': False, 'slot': 1}]
zygarde-10-power-construct
[{'ability': {'name': 'power-construct', 'url': 'https://pokeapi.co/api/v2/ability/211/'}, 'is_hidden': False, 'slot': 1}]
zygarde-50-power-construct
[{'ability': {'name': 'power-construct', 'url': 'https://pokeapi.co/api/v2/ability/211/'}, 'is_hidden': False, 'slot': 1}]
zygarde-complete
[{'ability': {'name': 'power-construct', 'url': 'https://pokeapi.co/api/v2/ability/211/'}, 'is_hidden': False, 'slot': 1}]
zygarde-10
[{'ability': {'name': 'aura-break', 'url': 'https://pokeapi.co/api/v2/ability/188/'}, 'is_hidden': False, 'slot': 1}]
zygarde-mega
[]


In [ ]:
# Combine zygarde-10 and zygarde-10-power-construct as well as zygarde-50 and zygarde-50-power-construct into single entries

z10ab = pokemon_raw['zygarde-10'].get('abilities')
z10ab.append({'ability': {'name': 'power-construct', 'url': 'https://pokeapi.co/api/v2/ability/211/'}, 'is_hidden': False, 'slot': 2})
z50ab = pokemon_raw['zygarde-50'].get('abilities')
z50ab.append({'ability': {'name': 'power-construct', 'url': 'https://pokeapi.co/api/v2/ability/211/'}, 'is_hidden': False, 'slot': 2})

In [ ]:
for pkmn, details in pokemon_raw.items():
    if('rockruff' in pkmn):
      print(pkmn)
      print(details.get('abilities'))

rockruff
[{'ability': {'name': 'keen-eye', 'url': 'https://pokeapi.co/api/v2/ability/51/'}, 'is_hidden': False, 'slot': 1}, {'ability': {'name': 'vital-spirit', 'url': 'https://pokeapi.co/api/v2/ability/72/'}, 'is_hidden': False, 'slot': 2}, {'ability': {'name': 'steadfast', 'url': 'https://pokeapi.co/api/v2/ability/80/'}, 'is_hidden': True, 'slot': 3}]
rockruff-own-tempo
[{'ability': {'name': 'own-tempo', 'url': 'https://pokeapi.co/api/v2/ability/20/'}, 'is_hidden': False, 'slot': 1}]


In [ ]:
for pkmn, details in pokemon_raw.items():
    if('minior' in pkmn):
      print(pkmn)

minior-red-meteor
minior-orange-meteor
minior-yellow-meteor
minior-green-meteor
minior-blue-meteor
minior-indigo-meteor
minior-violet-meteor
minior-red
minior-orange
minior-yellow
minior-green
minior-blue
minior-indigo
minior-violet


In [ ]:
for pkmn, details in pokemon_raw.items():
    if('mimikyu' in pkmn):
      print(pkmn)

mimikyu-disguised
mimikyu-busted
mimikyu-totem-disguised
mimikyu-totem-busted


In [ ]:
for pkmn, details in pokemon_raw.items():
    if('totem' in pkmn):
      print(pkmn)

raticate-totem-alola
gumshoos-totem
vikavolt-totem
lurantis-totem
salazzle-totem
mimikyu-totem-disguised
mimikyu-totem-busted
kommo-o-totem
marowak-totem
ribombee-totem
araquanid-totem
togedemaru-totem


In [ ]:
for pkmn, details in pokemon_raw.items():
    if('magearna' in pkmn):
      print(pkmn)

magearna
magearna-original
magearna-mega
magearna-original-mega


In [ ]:
for pkmn, details in pokemon_raw.items():
    if('cramorant' in pkmn):
      print(pkmn)

cramorant
cramorant-gulping
cramorant-gorging


In [ ]:
for pkmn, details in pokemon_raw.items():
    if('morpeko' in pkmn):
      print(pkmn)

morpeko-full-belly
morpeko-hangry


In [ ]:
for pkmn, details in pokemon_raw.items():
    if('eternatus' in pkmn):
      print(pkmn)

eternatus
eternatus-eternamax


In [ ]:
for pkmn, details in pokemon_raw.items():
    if('zarude' in pkmn):
      print(pkmn)

zarude
zarude-dada


In [ ]:
for pkmn, details in pokemon_raw.items():
    if('maushold' in pkmn):
      print(pkmn)

maushold-family-of-four
maushold-family-of-three


In [ ]:
for pkmn, details in pokemon_raw.items():
    if('tatsugiri' in pkmn):
      print(pkmn)

tatsugiri-curly
tatsugiri-droopy
tatsugiri-stretchy
tatsugiri-curly-mega
tatsugiri-droopy-mega
tatsugiri-stretchy-mega


In [ ]:
for pkmn, details in pokemon_raw.items():
    if('dudunsparce' in pkmn):
      print(pkmn)

dudunsparce-two-segment
dudunsparce-three-segment


In [ ]:
for pkmn, details in pokemon_raw.items():
    if('koraidon' in pkmn):
      print(pkmn)

koraidon
koraidon-limited-build
koraidon-sprinting-build
koraidon-swimming-build
koraidon-gliding-build


In [ ]:
for pkmn, details in pokemon_raw.items():
    if('miraidon' in pkmn):
      print(pkmn)

miraidon
miraidon-low-power-mode
miraidon-drive-mode
miraidon-aquatic-mode
miraidon-glide-mode


In [ ]:
to_be_removed = [
    "pikachu-rock-star", "pikachu-belle", "pikachu-pop-star", "pikachu-phd",
    "pikachu-libre", "pikachu-cosplay", "pikachu-original-cap", "pikachu-hoenn-cap",
    "pikachu-sinnoh-cap", "pikachu-unova-cap", "pikachu-kalos-cap", "pikachu-alola-cap",
    "pikachu-partner-cap", "pikachu-starter", "pikachu-world-cap", "greninja-battle-bond",
    "greninja-ash", "eevee-starter", "floette-eternal", "floette-mega",
    "rockruff-own-tempo", "minior-orange-meteor", "minior-yellow-meteor", "minior-green-meteor",
    "minior-blue-meteor", "minior-indigo-meteor", "minior-violet-meteor", "minior-orange",
    "minior-yellow", "minior-green", "minior-blue", "minior-indigo",
    "minior-violet", "mimikyu-busted", "raticate-totem-alola", "gumshoos-totem",
    "vikavolt-totem", "lurantis-totem", "salazzle-totem", "mimikyu-totem-disguised",
    "mimikyu-totem-busted", "kommo-o-totem", "marowak-totem", "ribombee-totem",
    "araquanid-totem", "togedemaru-totem", "magearna-original", "magearna-original-mega",
    "cramorant-gulping", "cramorant-gorging", "morpeko-hangry", "eternatus-eternamax",
    "zarude-dada", "maushold-family-of-three", "tatsugiri-droopy", "tatsugiri-stretchy",
    "tatsugiri-droopy-mega", "tatsugiri-stretchy-mega", "dudunsparce-three-segment", "koraidon-limited-build",
    "koraidon-sprinting-build", "koraidon-swimming-build", "koraidon-gliding-build", "miraidon-low-power-mode",
    "miraidon-drive-mode", "miraidon-aquatic-mode", "miraidon-glide-mode", "zygarde-10-power-construct", "zygarde-50-power-construct"
]

for key in to_be_removed:
  del pokemon_raw[key]

##Correct Generation Mapping

As explained previously, the generation field is obtained through the species endpoint and multiple Pokémon can be from the same species. The below mapping fixes false generation labels that arise from this problem. The reason these incorrect labels happen are the following:
### Reason 1: Regional Variants
Some Pokémon have regional variants, they are named after the region they have adapted to. Alola refers to the region in which generation 7 takes place, Hisui refers to a region that was released in generation 8, same for Galar. Paldea refers to the region in which generation 9 takes place.
### Reason 2: Alternative Forms (Mega Evolution, Gigantamax, Primal ,Origin)
Some Pokémon have these alternative for that are released in generations seperate from the Pokémon themselves. For example, Dialga and Palkia's origin forms were released in generation 8 whereas they themselves were released in 4.

In [ ]:
gen_map = {
    6: ["primal", "venusaur-mega", "charizard-mega-x", "charizard-mega-y", "blastoise-mega",
    "alakazam-mega", "gengar-mega", "kangaskhan-mega", "pinsir-mega",
    "gyarados-mega", "aerodactyl-mega", "mewtwo-mega-x", "mewtwo-mega-y",
    "ampharos-mega", "scizor-mega", "heracross-mega", "houndoom-mega",
    "tyranitar-mega", "blaziken-mega", "gardevoir-mega", "mawile-mega",
    "aggron-mega", "medicham-mega", "manectric-mega", "banette-mega",
    "absol-mega", "garchomp-mega", "lucario-mega", "abomasnow-mega",
    "beedrill-mega", "pidgeot-mega", "slowbro-mega", "steelix-mega",
    "sceptile-mega", "swampert-mega", "sableye-mega", "sharpedo-mega",
    "camerupt-mega", "altaria-mega", "glalie-mega", "salamence-mega",
    "metagross-mega", "latias-mega", "latios-mega", "rayquaza-mega",
    "lopunny-mega", "gallade-mega", "audino-mega", "diancie-mega"],
    7: ["alola"],
    8: ["galar", "hisui", "gmax", "dialga-origin", "palkia-origin", "white-striped"],
    9: ["paldea", "mega"]
}

In [ ]:
def gen_fix(current_pkmn, new_entry):
  caught = False
  if("mega-z" in current_pkmn):
    new_entry['generation'] = 9
  else:
    for gen_num, keywords in gen_map.items():
      for key in keywords:
          if key in current_pkmn:
            caught = True
            new_entry['generation'] = gen_num
            # print(f"{current_pkmn} gen corrected to {new_entry['generation']}")
            break
      if(caught):
        break

In [ ]:
processed_pokemon = {}

str_to_int_map = {
    "generation-i": 1,
    "generation-ii": 2,
    "generation-iii": 3,
    "generation-iv": 4,
    "generation-v": 5,
    "generation-vi": 6,
    "generation-vii": 7,
    "generation-viii": 8,
    "generation-ix": 9,
}

# Fields to keep from Pokemon
pkmn_fields = [
    'abilities', 'is_default', 'order', 'past_abilities',
    'past_stats', 'past_types', 'stats', 'types'
]

# Fields to keep from Species
species_fields = [
    'evolves_from_species', 'generation', 'is_baby',
    'is_legendary', 'is_mythical', 'gender_rate'
]

for pkmn_name, pkmn_data in pokemon_raw.items():
    # 1. Find the associated species
    linked_species_name = pkmn_data.get('species', {}).get('name')

    if linked_species_name in species_raw:
        spec_data = species_raw[linked_species_name]

        # 2. Extract Pokemon fields
        new_entry = {field: pkmn_data.get(field) for field in pkmn_fields}

        # 3. Add Species fields
        for field in species_fields:
            new_entry[field] = spec_data.get(field)

        # 4. Fix Gen info is necessary
        new_entry['generation'] = str_to_int_map[new_entry['generation']['name']]
        if("-" in pkmn_name):
          gen_fix(pkmn_name, new_entry)

        processed_pokemon[pkmn_name] = new_entry
    else:
        # Report missing species
        print(f"Species '{linked_species_name}' not found for Pokémon '{pkmn_name}'")

# 4. Save to Drive
output_path = os.path.join(project_data_path, 'pokemon.json')
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(processed_pokemon, f, indent=4)

print(f"File saved to: {output_path}")

File saved to: /content/drive/MyDrive/DSA210-Project/data/pokemon.json


In [ ]:
# Unnecessary fields: id, main_region, name, names, version_groups

print(generations_raw["generation-i"].keys())

dict_keys(['abilities', 'id', 'main_region', 'moves', 'name', 'names', 'pokemon_species', 'types', 'version_groups'])


In [ ]:
generations = {}

keys_to_remove_generations = [
    'id',
    'name',
    'names',
    'main_region',
    'version_groups'
]

for generation, gen_data in generations_raw.items():

      id = gen_data.get('id')
      clean_gen_data = gen_data.copy()

      # Remove the unwanted keys
      for key in keys_to_remove_generations:
          clean_gen_data.pop(key, None) # None prevents errors if key is missing

      generations[id] = clean_gen_data

# Manually remove the unknown type and shadow type

print(generations[2]["types"])
generations[2]["types"].pop()
print(generations[2]["types"])

print(generations[3]["types"])
generations[3]["types"].pop()
print(generations[3]["types"])

output_path = os.path.join(project_data_path, 'generations.json')
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(generations, f, indent=4)

print(f"File saved to: {output_path}")

[{'name': 'steel', 'url': 'https://pokeapi.co/api/v2/type/9/'}, {'name': 'dark', 'url': 'https://pokeapi.co/api/v2/type/17/'}, {'name': 'unknown', 'url': 'https://pokeapi.co/api/v2/type/10001/'}]
[{'name': 'steel', 'url': 'https://pokeapi.co/api/v2/type/9/'}, {'name': 'dark', 'url': 'https://pokeapi.co/api/v2/type/17/'}]
[{'name': 'shadow', 'url': 'https://pokeapi.co/api/v2/type/10002/'}]
[]
File saved to: /content/drive/MyDrive/DSA210-Project/data/generations.json


In [ ]:
# Unnecessary fields: game_indices, generation, id, name, names, sprites

print(types_raw["normal"].keys())

dict_keys(['damage_relations', 'game_indices', 'generation', 'id', 'move_damage_class', 'moves', 'name', 'names', 'past_damage_relations', 'pokemon', 'sprites'])


In [ ]:
types = {}

keys_to_remove_types = [
    'id',
    'name',
    'names',
    'game_indices',
    'generation',
    'sprites'
]

types_to_remove = ['stellar', 'unknown', 'shadow']

for pkmn_type, type_data in types_raw.items():

      clean_type_data = type_data.copy()

      # Remove the unwanted keys
      for key in keys_to_remove_types:
          clean_type_data.pop(key, None) # None prevents errors if key is missing

      types[pkmn_type] = clean_type_data

for key in types_to_remove:
  del types[key]

output_path = os.path.join(project_data_path, 'types.json')
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(types, f, indent=4)

print(f"File saved to: {output_path}")

File saved to: /content/drive/MyDrive/DSA210-Project/data/types.json
